## **Notebook 04 — Chunker**
- Load parsed markdown → split into parent + child chunks → attach metadata → embed with BGE-M3 → store in pgvector.

In [1]:
import os, json, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional
from dotenv import load_dotenv

load_dotenv()
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")

print("Imports OK")

Imports OK


In [2]:
parsed_path = Path("../data/parsed/test_document.pdf.json")

with open(parsed_path, "r", encoding="utf-8") as f:
    parsed = json.load(f)

markdown  = parsed["markdown"]
filename  = parsed["filename"]
num_pages = parsed["num_pages"]

print(f"Filename  : {filename}")
print(f"Pages     : {num_pages}")
print(f"Markdown  : {len(markdown)} characters")
print(f"Preview   :")
print(markdown[:300])

Filename  : test_document.pdf
Pages     : 55
Markdown  : 156703 characters
Preview   :
## Preamble

Bangladesh Red Crescent Society (BDRCS) has long been awaiting a well-documented Human  Resource  Policy.  The  HR  Policy  is  one  of  the  two  outcomes  of  a  long  and tedious  journey  of  over  five  months  undertaken  by  many  people  of  different  levels representing BDRCS 


In [3]:
@dataclass
class Chunk:
    content        : str
    parent_content : Optional[str]   # None for parent chunks
    chunk_type     : str             # "child" or "parent"
    chunk_index    : int
    section        : str
    metadata       : dict = field(default_factory=dict)

# We store BOTH parent and child chunks in the DB
# Retrieval searches child chunks (precise)
# Generation receives parent chunks (full context)
print("Chunk dataclass defined")

Chunk dataclass defined


In [4]:
def detect_section(text: str) -> str:
    """
    Walk backwards through lines to find the nearest ## heading.
    This becomes the section tag on every chunk.
    """
    for line in reversed(text.split("\n")):
        line = line.strip()
        if line.startswith("## "):
            return line.lstrip("#").strip()
        if line.startswith("# "):
            return line.lstrip("#").strip()
    return "general"


# Test it
test_snippet = """## Leave Policy
All employees are entitled to 15 days annual leave."""

print("Section detected:", detect_section(test_snippet))

Section detected: Leave Policy


In [5]:
import re

def split_into_sentences(text: str) -> List[str]:
    """
    Split text into sentences. Handles:
    - Bengali text (ends with ।)
    - English text (ends with . ! ?)
    - Skips markdown headers and empty lines
    """
    sentences = []
    for line in text.split("\n"):
        line = line.strip()

        # Skip empty lines and pure markdown headers
        if not line or line.startswith("#"):
            continue

        # Split on sentence endings — handles English + Bengali
        parts = re.split(r'(?<=[.!?।])\s+', line)
        for part in parts:
            part = part.strip()
            if len(part) > 20:   # skip very short fragments
                sentences.append(part)

    return sentences


sentences = split_into_sentences(markdown)
print(f"Total sentences found: {len(sentences)}")
print("\nFirst 5 sentences:")
for i, s in enumerate(sentences[:5], 1):
    print(f"  {i}. {s[:90]}")

Total sentences found: 921

First 5 sentences:
  1. Bangladesh Red Crescent Society (BDRCS) has long been awaiting a well-documented Human  Re
  2. The  HR  Policy  is  one  of  the  two  outcomes  of  a  long  and tedious  journey  of  o
  3. Broadly speaking, the assignment included two tasks simultaneously, the development of a f
  4. Therefore,  it  is  the  request  of  the  authors  to  read  this  document  in conjuncti
  5. Despite this being a gigantic task, collection of relevant information has been completed 


In [6]:
def build_chunks(
    markdown    : str,
    filename    : str,
    child_size  : int = 3,    # sentences per child chunk
    parent_size : int = 7,    # sentences per parent chunk
    overlap     : int = 1,    # sentence overlap between chunks
) -> List[Chunk]:
    """
    Parent-child chunking strategy:
    - Child chunks  = small, precise (3 sentences) — used for retrieval
    - Parent chunks = larger context (7 sentences) — sent to LLM
    Each child knows its parent content via parent_content field.
    """
    sentences = split_into_sentences(markdown)
    chunks    = []
    idx       = 0

    i = 0
    while i < len(sentences):

        # ── Parent chunk ──────────────────────────────
        parent_sents   = sentences[i : i + parent_size]
        parent_content = " ".join(parent_sents)
        section        = detect_section(
                            "\n".join(markdown.split("\n")[:i+1])
                         )

        # ── Child chunks inside this parent ───────────
        j = 0
        while j < len(parent_sents):
            child_sents   = parent_sents[j : j + child_size]
            child_content = " ".join(child_sents)

            if len(child_content.strip()) > 30:
                chunk = Chunk(
                    content        = child_content,
                    parent_content = parent_content,
                    chunk_type     = "child",
                    chunk_index    = idx,
                    section        = section,
                    metadata       = {
                        "filename"    : filename,
                        "section"     : section,
                        "chunk_type"  : "child",
                        "chunk_index" : idx,
                        "parent_size" : len(parent_sents),
                    }
                )
                chunks.append(chunk)
                idx += 1

            j += child_size - overlap   # overlap by 1 sentence

        i += parent_size - overlap      # slide window with overlap

    return chunks


chunks = build_chunks(markdown, filename)

print(f"Total chunks created : {len(chunks)}")
print(f"\nSample child chunk:")
print(f"  Content        : {chunks[0].content[:120]}")
print(f"  Parent preview : {chunks[0].parent_content[:120]}")
print(f"  Section        : {chunks[0].section}")
print(f"  Metadata       : {chunks[0].metadata}")

Total chunks created : 610

Sample child chunk:
  Content        : Bangladesh Red Crescent Society (BDRCS) has long been awaiting a well-documented Human  Resource  Policy. The  HR  Polic
  Parent preview : Bangladesh Red Crescent Society (BDRCS) has long been awaiting a well-documented Human  Resource  Policy. The  HR  Polic
  Section        : Preamble
  Metadata       : {'filename': 'test_document.pdf', 'section': 'Preamble', 'chunk_type': 'child', 'chunk_index': 0, 'parent_size': 7}


In [7]:
from collections import Counter

section_counts = Counter(c.section for c in chunks)

print("Chunks per section:\n")
for section, count in section_counts.most_common():
    bar = "█" * (count // 2)
    print(f"  {section:<35} {count:>3}  {bar}")


Chunks per section:

  Table of Contents                    72  ████████████████████████████████████
  1.8 Functional Definitions           24  ████████████
  2.3 The Organizational Structure (Organogram) BDRCS  24  ████████████
  Abbreviations                        16  ████████
  Leave Without Pay/Extraordinary Leave  16  ████████
  5.9 Traveling and Daily Allowances   16  ████████
  Seven Staff Retention and Reward     16  ████████
  4.6 Appointment                      15  ███████
  3.2 Types of Employees               12  ██████
  5.2 Remuneration                     12  ██████
  Privilege Leave                      12  ██████
  5.4 Promotion                        12  ██████
  b) Rate of Gratuity                  12  ██████
  6.1.1 Principles and definitions     11  █████
  1.1 What this Policy is all about     8  ████
  1.3 Present Governance Structure      8  ████
  2.2 Creation of a New Research &amp; Communication Department   8  ████
  3.1 Analyzing Existing Staff Strength  

In [8]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    use_fp16=False,
    cache_dir="../models/bge-m3"
)

# Extract all child content for batch encoding
texts = [c.content for c in chunks]

print(f"Encoding {len(texts)} chunks with BGE-M3...")

output = model.encode(
    texts,
    batch_size=16,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)

dense_vecs     = output["dense_vecs"]       # shape: (N, 1024)
sparse_weights = output["lexical_weights"]  # list of dicts

print(f"Dense shape    : {dense_vecs.shape}")
print(f"Sparse sample  : {len(sparse_weights[0])} unique tokens in chunk 0")


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Encoding 610 chunks with BGE-M3...


pre tokenize: 100%|██████████| 39/39 [00:00<00:00, 1095.42it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 39/39 [00:05<00:00,  6.58it/s]

Dense shape    : (610, 1024)
Sparse sample  : 83 unique tokens in chunk 0


In [10]:
import psycopg2, json as jsonlib
from pgvector.psycopg2 import register_vector

conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()

# Create a document record
cur.execute("""
    INSERT INTO documents (filename, file_type, status)
    VALUES (%s, %s, 'ready')
    RETURNING id;
""", (filename, "pdf"))
doc_id = cur.fetchone()[0]
print(f"Document ID : {doc_id}")

# Insert every chunk
inserted = 0
for chunk, vec, sparse in zip(chunks, dense_vecs, sparse_weights):

    # Add sparse weights to metadata for later BM25s index build
    meta = {**chunk.metadata, "sparse_weights": sparse}

    cur.execute("""
        INSERT INTO chunks
            (document_id, content, parent_content, embedding, metadata)
        VALUES (%s, %s, %s, %s, %s::jsonb)
    """, (
        doc_id,
        chunk.content,
        chunk.parent_content,
        vec.tolist(),
        jsonlib.dumps(meta, default=float),
    ))
    inserted += 1

conn.commit()

# Verify
cur.execute("SELECT COUNT(*) FROM chunks WHERE document_id = %s;", (doc_id,))
count = cur.fetchone()[0]
print(f"Chunks stored : {count}")
print(f"Inserted      : {inserted}")

Document ID : 22238a3b-7c76-4d3a-947f-ce1cf97d6fbd
Chunks stored : 610
Inserted      : 610


In [11]:
# Test that we can actually retrieve the chunks we just stored

query     = "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"
query_out = model.encode([query], return_dense=True)
query_vec = query_out["dense_vecs"][0]

cur.execute("""
    SELECT
        content,
        parent_content,
        metadata->>'section'  AS section,
        1 - (embedding <=> %s::vector) AS score
    FROM chunks
    WHERE document_id = %s
    ORDER BY embedding <=> %s::vector
    LIMIT 3;
""", (query_vec.tolist(), doc_id, query_vec.tolist()))

results = cur.fetchall()

print(f'Query: "{query}"\n')
for rank, (content, parent, section, score) in enumerate(results, 1):
    print(f"Rank {rank} | Score: {score:.4f} | Section: {section}")
    print(f"  Child  : {content[:100]}")
    print(f"  Parent : {parent[:100]}")
    print()

conn.close()

Query: "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

Rank 1 | Score: 0.6662 | Section: 3.3 Internal Candidate
  Child  : Permanent/regular: Permanent or regular employee shall be the one who is employed to work until his/
  Parent : Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary period who

Rank 2 | Score: 0.6566 | Section: 3.3 Internal Candidate
  Child  : Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary period who
  Parent : Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary period who

Rank 3 | Score: 0.6287 | Section: b) Rate of Gratuity
  Child  : - -The candidate must have completed at least two years of continuous service with BDRCS before the 
  Parent : - -The candidate must have completed at least one year of continuous service with BDRCS before the t



In [13]:
for rank, (content, parent, section, score) in enumerate(results, 1):
    print(f"Rank {rank} | Score: {score:.4f} | Section: {section}")
    print(f"  Child  : {content[:1000]}")
    print(f"  Parent : {parent[:1000]}")
    print()

Rank 1 | Score: 0.6662 | Section: 3.3 Internal Candidate
  Child  : Permanent/regular: Permanent or regular employee shall be the one who is employed to work until his/her retirement age, if not otherwise retrenched or terminated for any gross anomalies or violation  of  Code  of  Conduct  of  the  organization. He/she  shall  enjoy  all  entitlements  that BDRCS offers  him/her  as  described  in  this  HR  Policy  Guideline  document. Upon  successful completion of the probationary period, the person concerned shall be regarded as permanent/regular staff member.
  Parent : Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary period who are not considered as staff members, but employees definitely. Below are brief descriptions of various categories of employees: Permanent/regular: Permanent or regular employee shall be the one who is employed to work until his/her retirement age, if not otherwise retrenched or terminated for any gross anomalies or 

In [14]:
print("=" * 48)
print("STEP 5 COMPLETE")
print("=" * 48)
print(f"  Parsed markdown loaded           : OK")
print(f"  Sentences split                  : OK")
print(f"  Parent-child chunks built        : {len(chunks)} chunks")
print(f"  BGE-M3 embeddings created        : {dense_vecs.shape}")
print(f"  Chunks stored in pgvector        : OK")
print(f"  Similarity search verified       : OK")
print()
print("  Key insight:")
print("  Child chunk = precise retrieval target")
print("  Parent chunk = full context for the LLM")
print()
print("READY FOR STEP 6 — BM25s Index")

STEP 5 COMPLETE
  Parsed markdown loaded           : OK
  Sentences split                  : OK
  Parent-child chunks built        : 610 chunks
  BGE-M3 embeddings created        : (610, 1024)
  Chunks stored in pgvector        : OK
  Similarity search verified       : OK

  Key insight:
  Child chunk = precise retrieval target
  Parent chunk = full context for the LLM

READY FOR STEP 6 — BM25s Index


In [15]:
# Search your raw markdown for the retirement age answer
lines = markdown.split("\n")
print("Lines containing 'retirement' or '60':\n")
for i, line in enumerate(lines):
    if "retirement" in line.lower() or "retire" in line.lower() or "60" in line:
        print(f"  Line {i:>4}: {line.strip()}")


Lines containing 'retirement' or '60':

  Line   57: | 3.8                                                                                                                                                 | Retirement .........................................................................................................................15               |
  Line  305: Permanent/regular: Permanent or regular employee shall be the one who is employed to work until his/her retirement age, if not otherwise retrenched or terminated for any gross anomalies or violation  of  Code  of  Conduct  of  the  organization.  He/she  shall  enjoy  all  entitlements  that BDRCS offers  him/her  as  described  in  this  HR  Policy  Guideline  document.  Upon  successful completion of the probationary period, the person concerned shall be regarded as permanent/regular staff member.
  Line  347: ## 3.8 Retirement
  Line  349: The staff members of regular/permanent positions in the society shall retire on th